In [1]:
def get_matches_today_data():
    import statsapi
    import mlbstatsapi
    from datetime import datetime

    # Get today's schedule
    matches_today = []

    # get the proper formatted date
    mlb_date = datetime.now().strftime("%m/%d/%Y")

    # get the schedule as a dictionary for today
    schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

    # iterate through each game of the schedule
    for x in schedule:

        # initialize game data dictionary
        game_data = {}
        
        # away_name
        game_data.update({'away_name': x.get('away_name')}) 
        
        # home_name
        game_data.update({'home_name': x.get('home_name')})
        
        # away_id
        game_data.update({'away_id': x.get('away_id')})

        away_team_leaders_hr = []
        # add top away team guys here
        away_leaders = statsapi.team_leader_data(x.get('away_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in away_leaders:
            away_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'away_team_leaders_hr': away_team_leaders_hr})

        home_team_leaders_hr = []
        # add top away team guys here
        home_leaders = statsapi.team_leader_data(x.get('home_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in home_leaders:
            home_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'home_team_leaders_hr': home_team_leaders_hr})

        # home_id
        game_data.update({'home_id': x.get('home_id')})
        # home_probable_pitcher
        game_data.update({'home_probable_pitcher': x.get('home_probable_pitcher')})
        # away_probable_pitcher
        game_data.update({'away_probable_pitcher': x.get('away_probable_pitcher')})

        matches_today.append(game_data)


    mlb = mlbstatsapi.Mlb()

    for x in matches_today:
  
        away_probable_pitcher = x.get('away_probable_pitcher')
        
        # Check if away_probable_pitcher is valid
        if not away_probable_pitcher:
            print(f"Warning: Missing away_probable_pitcher for game: {x}")
            continue  # Skip this game if no pitcher is available

        pitcher_ids = mlb.get_people_id(away_probable_pitcher)
        
        # Check if pitcher_ids is not empty
        if not pitcher_ids:
            print(f"Warning: No pitcher ID found for {away_probable_pitcher}")
            continue  # Skip this game if no pitcher ID is found

        pitcher_id = pitcher_ids[0]  # Safely access the first element

        BvP = []
        for y in x.get('home_team_leaders_hr', []):  # Default to an empty list if key is missing
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    bvp_matchup = f"pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}"
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    dict2.update({'pitcher': p_id.__dict__.get('fullname')})
                    dict2.update({'batter': b_id.__dict__.get('fullname')})
                    BvP.append(dict2)

            except KeyError as e:
                print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
            except Exception as e:
                print(f"Unexpected error: {e}. Skipping this player.")
        
        
        home_probable_pitcher = x.get('home_probable_pitcher')
        
        # Check if home_probable_pitcher is valid
        if not home_probable_pitcher:
            print(f"Warning: Missing home_probable_pitcher for game: {x}")
            continue  # Skip this game if no pitcher is available

        pitcher_ids = mlb.get_people_id(home_probable_pitcher)
        
        # Check if pitcher_ids is not empty
        if not pitcher_ids:
            print(f"Warning: No pitcher ID found for {home_probable_pitcher}")
            continue  # Skip this game if no pitcher ID is found

        pitcher_id = pitcher_ids[0]  # Safely access the first element

        for y in x.get('away_team_leaders_hr', []):  # Default to an empty list if key is missing
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    bvp_matchup = f"pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}"
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    dict2.update({'pitcher': p_id.__dict__.get('fullname')})
                    dict2.update({'batter': b_id.__dict__.get('fullname')})
                    BvP.append(dict2)

            except KeyError as e:
                print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
            except Exception as e:
                print(f"Unexpected error: {e}. Skipping this player.")
        
        # Add the BvP stats to the matches_today dictionary
        x.update({'BvP_stats': BvP})

    return matches_today

# todays_matches = get_matches_today_data()

# import sys
# import os


# sys.stdout = open(os.devnull, 'w')

# Call your function
todays_matches = get_matches_today_data()

# # Restore output
# sys.stdout = sys.__stdout__

# # Print a readable output of each item in the dictionary of the list matches_today
# for match in todays_matches:
#     print(f"Match: AWAY: {match['away_name']} vs HOME: {match['home_name']}")
#     print(f"Away Team ID: {match['away_id']}, Home Team ID: {match['home_id']}")
#     print(f"Away Probable Pitcher: {match['away_probable_pitcher']}")
#     print(f"Home Probable Pitcher: {match['home_probable_pitcher']}")
    
#     print("Away Team Leaders in Home Runs:")
#     for leader in match['away_team_leaders_hr']:
#         print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
#     print("Home Team Leaders in Home Runs:")
#     for leader in match['home_team_leaders_hr']:
#         print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
#     if 'BvP_stats' in match:
#         print("Batter vs Pitcher Stats:")
#         for bvp in match['BvP_stats']:
#             print(f"  Matchup: {bvp['bvp_matchup']}")
#             for stat, value in bvp['bvp_stats'].items():
#                 print(f"    {stat}: {value}")
    
#     print("\n")  # Print a newline for better readability between matches



https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/682829/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/682829
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/669720/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/669720
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/680574/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/680574
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/682622/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/682622
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/642851/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/

In [2]:

for match in todays_matches:

    if 'BvP_stats' in match:

        print("Batter vs Pitcher Stats:")
        for bvp in match['BvP_stats']:
            print()
            # print(f"Match: AWAY: {match['away_name']} vs HOME: {match['home_name']}")
            # print(f"Away Probable Pitcher: {match['away_probable_pitcher']}")
            # print(f"Home Probable Pitcher: {match['home_probable_pitcher']}")
            away = match['away_probable_pitcher']
            home = match['home_probable_pitcher']
            batter_name = bvp['bvp_matchup']
            if bvp['pitcher'] == away:
                print(f"Away Pitcher:  {bvp['pitcher']:<17} {match['away_name']:<25}")
                beans = 'away'
            elif bvp['pitcher'] == home:
                print(f"Home Pitcher:  {bvp['pitcher']:<17} {match['home_name']:<25}")
                beans = 'home'
            if beans == 'away':
                print(f"Home  Batter:  {bvp['batter']:<17} {match['home_name']:<25}")
            elif beans == 'home':
                print(f"Away  Batter:  {bvp['batter']:<17} {match['away_name']:<25}")
            # print(f"Pitcher: {bvp['pitcher']:>10}")
            print(f" AB: {bvp['bvp_stats'].get('atbats', 'N/A'):>7}")
            print(f"  H: {bvp['bvp_stats'].get('hits', 'N/A'):>7}")
            print(f" HR: {bvp['bvp_stats'].get('homeruns', 'N/A'):>7}")
            print(f"AVG: {bvp['bvp_stats'].get('avg', 'N/A'):>7}")
            print(f"RBI: {bvp['bvp_stats'].get('rbi', 'N/A'):>7}")
            print(f"obp: {bvp['bvp_stats'].get('obp', 'N/A'):>7}")
            print(f"ops: {bvp['bvp_stats'].get('ops', 'N/A'):>7}")
            print()

    print("\n")  # Print a newline for better readability between matches


Batter vs Pitcher Stats:

Away Pitcher:  Miles Mikolas     St. Louis Cardinals      
Home  Batter:  Elly De La Cruz   Cincinnati Reds          
 AB:      11
  H:       4
 HR:       0
AVG:    .364
RBI:       3
obp:    .417
ops:   1.053


Away Pitcher:  Miles Mikolas     St. Louis Cardinals      
Home  Batter:  Austin Hays       Cincinnati Reds          
 AB:       3
  H:       0
 HR:       0
AVG:    .000
RBI:       0
obp:    .000
ops:    .000


Away Pitcher:  Miles Mikolas     St. Louis Cardinals      
Home  Batter:  Matt McLain       Cincinnati Reds          
 AB:       6
  H:       2
 HR:       0
AVG:    .333
RBI:       0
obp:    .333
ops:   1.000


Away Pitcher:  Miles Mikolas     St. Louis Cardinals      
Home  Batter:  Noelvi Marte      Cincinnati Reds          
 AB:       6
  H:       2
 HR:       0
AVG:    .333
RBI:       1
obp:    .333
ops:    .833


Away Pitcher:  Miles Mikolas     St. Louis Cardinals      
Home  Batter:  Jeimer Candelario Cincinnati Reds          
 AB:       4

In [ ]:
# WRITE BVP TO A FILE

import os

# Ensure the "text_output" folder exists
os.makedirs("text_output", exist_ok=True)

# Open the file in write mode
with open("text_output/BVP.txt", "w") as file:
    for match in todays_matches:

        if 'BvP_stats' in match:

            file.write("Batter vs Pitcher Stats:\n")
            for bvp in match['BvP_stats']:
                file.write("\n")
                away = match['away_probable_pitcher']
                home = match['home_probable_pitcher']
                batter_name = bvp['bvp_matchup']
                if bvp['pitcher'] == away:
                    file.write(f"Away Pitcher:  {bvp['pitcher']:<17} {match['away_name']:<25}\n")
                    beans = 'away'
                elif bvp['pitcher'] == home:
                    file.write(f"Home Pitcher:  {bvp['pitcher']:<17} {match['home_name']:<25}\n")
                    beans = 'home'
                if beans == 'away':
                    file.write(f"Home  Batter:  {bvp['batter']:<17} {match['home_name']:<25}\n")
                elif beans == 'home':
                    file.write(f"Away  Batter:  {bvp['batter']:<17} {match['away_name']:<25}\n")
                file.write(f" AB: {bvp['bvp_stats'].get('atbats', 'N/A'):>7}\n")
                file.write(f"  H: {bvp['bvp_stats'].get('hits', 'N/A'):>7}\n")
                file.write(f" HR: {bvp['bvp_stats'].get('homeruns', 'N/A'):>7}\n")
                file.write(f"AVG: {bvp['bvp_stats'].get('avg', 'N/A'):>7}\n")
                file.write(f"RBI: {bvp['bvp_stats'].get('rbi', 'N/A'):>7}\n")
                file.write(f"obp: {bvp['bvp_stats'].get('obp', 'N/A'):>7}\n")
                file.write(f"ops: {bvp['bvp_stats'].get('ops', 'N/A'):>7}\n")
                file.write("\n")

            file.write("\n")  # Write a newline for better readability between matches

In [ ]:

# Standing and schedule into text file
import statsapi
from datetime import datetime, timedelta

# Get current date
mlb_date = datetime.now().strftime("%m/%d/%Y")
file_date = datetime.now().strftime("%Y-%m-%d")


import os
from datetime import datetime, timedelta
import statsapi

# File name with current date
file_date = datetime.now().strftime("%Y-%m-%d")
file_name = f"MLB STANDINGS AND SCHEDULE.txt"

# Get yesterday's schedule
oneday = timedelta(days=1)
yesterday = datetime.now().date() - oneday
yschedule = statsapi.schedule(start_date=yesterday, end_date=yesterday)

# Get today's schedule
mlb_date = datetime.now().strftime("%m/%d/%Y")
schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

# Prepare content to write
yesterday_schedule_content = "\nYesterday's Schedule:\n" + "\n".join(
    f'{x.get("summary")}\n\n{statsapi.linescore(x.get("game_id"))}\n\n{statsapi.game_scoring_plays(x.get("game_id"))}\n\n-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*\n' for x in yschedule
)

standings_content = (
    "MLB Standings:\n"
    + statsapi.standings(leagueId=103, date=mlb_date)
    + statsapi.standings(leagueId=104, date=mlb_date)
)
today_schedule_content = "Today's Schedule:\n" + "\n".join(
    f'{x.get("summary")}' for x in schedule
)
print(yesterday_schedule_content)
print('')
print(standings_content)
print('')
print(today_schedule_content)




Yesterday's Schedule:
2025-04-29 - Minnesota Twins (1) @ Cleveland Guardians (2) (Final)

Final     1 2 3 4 5 6 7 8 9  R   H   E  
Twins     0 0 0 0 1 0 0 0 0  1   7   0  
Guardians 0 0 1 0 0 0 0 0 1  2   6   0  

Bo Naylor homers (3) on a fly ball to right field.
Bottom 3 - Minnesota Twins: 0, Cleveland Guardians: 1

Ty France homers (3) on a fly ball to right center field.
Top 5 - Minnesota Twins: 1, Cleveland Guardians: 1

Kyle Manzardo homers (8) on a fly ball to right field.
Bottom 9 - Minnesota Twins: 1, Cleveland Guardians: 2

-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*

2025-04-29 - New York Yankees (15) @ Baltimore Orioles (3) (Final)

Final   1 2 3 4 5 6 7 8 9  R   H   E  
Yankees 5 1 0 3 3 0 1 1 1  15  19  0  
Orioles 0 0 0 0 0 1 1 0 1  3   3   3  

Trent Grisham homers (8) on a fly ball to right field.
Top 1 - New York Yankees: 1, Baltimore Orioles: 0

Aaron Judge homers (9) on a fly ball to right field.
Top 1 - New York Yankees: 2, Baltimore Orioles: 0

Ben Rice homers (7) on a fly ba

In [9]:
# Standing and schedule into text file
import statsapi
from datetime import datetime, timedelta
import os

# Ensure the "text_output" folder exists
os.makedirs("text_output", exist_ok=True)

# File name with current date
file_date = datetime.now().strftime("%Y-%m-%d")
file_name = "Todays_Report.txt"

# Get yesterday's schedule
oneday = timedelta(days=1)
yesterday = datetime.now().date() - oneday
yschedule = statsapi.schedule(start_date=yesterday, end_date=yesterday)

# Get today's schedule
mlb_date = datetime.now().strftime("%m/%d/%Y")
schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

# Prepare content to write
yesterday_schedule_content = "\nYesterday's Schedule:\n" + "\n".join(
    f'{x.get("summary")}\n\n{statsapi.linescore(x.get("game_id"))}\n\n{statsapi.game_scoring_plays(x.get("game_id"))}\n\n-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*\n' for x in yschedule
)

standings_content = (
    "MLB Standings:\n"
    + statsapi.standings(leagueId=103, date=mlb_date)
    + statsapi.standings(leagueId=104, date=mlb_date)
)
today_schedule_content = "Today's Schedule:\n" + "\n".join(
    f'{x.get("summary")}' for x in schedule
)

# Combine all content
full_content = (
    yesterday_schedule_content + "\n\n" +
    standings_content + "\n\n" +
    today_schedule_content
)

# Write content to the file
with open(f"text_output/{file_name}", "w") as file:
    file.write(full_content)

print(f"Report saved to text_output/{file_name}")

Report saved to text_output/Todays_Report.txt


In [10]:
import os

# Ensure the "docs" folder exists
os.makedirs("docs", exist_ok=True)

# File paths
todays_report_path = "text_output/Todays_Report.txt"
bvp_path = "text_output/BVP.txt"
output_html_path = "docs/index.html"

# Read the contents of the text files
with open(todays_report_path, "r") as todays_report_file:
    todays_report_content = todays_report_file.read()

with open(bvp_path, "r") as bvp_file:
    bvp_content = bvp_file.read()

# Create the HTML content
html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>MLB Report</title>
</head>
<body>
    <ul>
    <li><a href='https://www.fantasyalarm.com/mlb/lineups'>BVP checker</a></li>
    <li><a href='https://www.baseball-reference.com'>baseball-reference</a></li>
    <li><a href='https://baseballsavant.mlb.com'>baseball-savant</a></li>
    <li><a href='https://www.fangraphs.com'>fangraphs</a></li>
    <li><a href='https://www.statmuse.com/mlb'>Stat muse</a></li>
    </ul>
    <h1>MLB Report</h1>
    <h2>Today's Report</h2>
    <pre>{todays_report_content}</pre>
    <h2>Batter vs Pitcher Stats</h2>
    <pre>{bvp_content}</pre>
</body>
</html>
"""

# Write the HTML content to the output file
with open(output_html_path, "w") as output_file:
    output_file.write(html_content)

print(f"HTML file saved to {output_html_path}")

HTML file saved to docs/index.html
